In [5]:
# Read the file from the Files section
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://962bc3af-b163-4e9a-9aa7-6f819f2845b1@onelake.dfs.fabric.microsoft.com/5d568980-3d51-4784-8ad7-aac6f9087f15/Files/CaseStudy5/Customer_Master.csv")

# Write it as a managed or external Delta table
df.write.format("delta").mode("overwrite").saveAsTable("GlobalCorp.Customer_Master")

StatementMeta(, e771a214-c83b-4041-a961-eb93ddb435b9, 7, Finished, Available, Finished, False)

In [8]:
from pyspark.sql import functions as F

StatementMeta(, d671fe8f-3537-4656-a708-e2dbe8c8037e, 10, Finished, Available, Finished, False)

In [7]:
cust_df = spark.read.table("LH_Nanda.GlobalCorp.customer_master")
txn_df = spark.read.csv("Files/CaseStudy5/AR_Transactions.csv",header=True,inferSchema=True)

StatementMeta(, d671fe8f-3537-4656-a708-e2dbe8c8037e, 9, Finished, Available, Finished, False)

In [21]:
display(txn_df)

StatementMeta(, e771a214-c83b-4041-a961-eb93ddb435b9, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e71f7470-b073-45c6-ae06-3d79a13646c1)

In [18]:
display(cust_df)

StatementMeta(, e771a214-c83b-4041-a961-eb93ddb435b9, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1eaa3465-7b39-4920-a25e-4fb92673000a)

In [7]:
filtered_df = cust_df.filter(F.col("CustomerID")=="10001")

StatementMeta(, 1b71d2a6-2073-4dfc-843d-5bb6a5ee7056, 9, Finished, Available, Finished, False)

In [8]:
print(filtered_df)

StatementMeta(, 1b71d2a6-2073-4dfc-843d-5bb6a5ee7056, 10, Finished, Available, Finished, False)

DataFrame[CustomerID: int, CustomerName: string, Address: string, Credit_Limit: double]


In [5]:
txn_df.filter(F.col("transactionAmount")<0)
# narrow = operates row by row on individual partitions without exchangingdata across nodes
txn_df.explain()

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 7, Finished, Available, Finished, False)

== Physical Plan ==
FileScan csv [TransactionID#781,CustomerID#782,Transaction_Date#783,TransactionAmount#784] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[abfss://962bc3af-b163-4e9a-9aa7-6f819f2845b1@onelake.dfs.fabric.micros..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<TransactionID:int,CustomerID:int,Transaction_Date:date,TransactionAmount:double>




In [6]:
txn_df.groupBy("Transaction_Date").count()
#wide - requires shuffling data across partitions so that all rows with the same transactiondate end up on the same nodeto compute the count

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 8, Finished, Available, Finished, False)

DataFrame[Transaction_Date: date, count: bigint]

In [7]:
txn_df.orderBy(F.col("TransactionAmount").desc())
#wide - requires a global sort across the entire dataset,which mandates a ata schuffle between partitions

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 9, Finished, Available, Finished, False)

DataFrame[TransactionID: int, CustomerID: int, Transaction_Date: date, TransactionAmount: double]

In [8]:
txn_df.sortWithinPartitions("TransactionAmount")
#narrow - sorts data locally within each partition independently,avoiding any data movement or shuffle across network nodes

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 10, Finished, Available, Finished, False)

DataFrame[TransactionID: int, CustomerID: int, Transaction_Date: date, TransactionAmount: double]

In [10]:
txn_df.repartition(50)
#wide - performs a full hash repartitioning of the data across 50 new partitions,causing a full data shuffle across netwok nodes

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 12, Finished, Available, Finished, False)

DataFrame[TransactionID: int, CustomerID: int, Transaction_Date: date, TransactionAmount: double]

In [12]:
txn_df.coalesce(1)
# narrow  - combines existing partitions on the same worker nodes without triggering a full cross network data shuffle

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 14, Finished, Available, Finished, False)

DataFrame[TransactionID: int, CustomerID: int, Transaction_Date: date, TransactionAmount: double]

In [13]:
txn_df.union(txn_df)
# narrow - merges partition refrences from both dataframes wihout moving rows across existing partitions

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 15, Finished, Available, Finished, False)

DataFrame[TransactionID: int, CustomerID: int, Transaction_Date: date, TransactionAmount: double]

In [14]:
txn_df.agg(F.sum("TransactionAmount"))
# wide - computes a global aggregate requiring values across all partitions to be combined ,which involves a shuffle step.

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 16, Finished, Available, Finished, False)

DataFrame[sum(TransactionAmount): double]

In [20]:
txn_df.join(F.broadcast(cust_df),"CustomerID")
# narrow - because cust_df is broadcast to every node via a broadcast hash join, the join is performed locally without shuffling txn_df
display(txn_df.join(F.broadcast(cust_df),"CustomerID"))

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ae7adba2-0499-4435-9cf2-11ff42e30f34)

In [21]:
txn_df.join(cust_df.hint("merge"),"CustomerID")
# wide - perform a sort merge join requiring both dataframes to be repartitioned and sorted on the join key(customerID) via a data shuffle

StatementMeta(, 03c15768-407a-4ab2-8206-1cb27fd720b5, 23, Finished, Available, Finished, False)

DataFrame[CustomerID: int, TransactionID: int, Transaction_Date: date, TransactionAmount: double, CustomerName: string, Address: string, Credit_Limit: double]

In [5]:
filtered_df = cust_df.filter(F.col("CustomerID")=="10001")
filtered_df.explain(mode="Formatted")

StatementMeta(, d671fe8f-3537-4656-a708-e2dbe8c8037e, 7, Finished, Available, Finished, False)

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet spark_catalog.chimcobldhq2ahridtqn081m8cikoi2v9pgmsp314l3morr2c5m46rrie0.customer_master (1)


(1) Scan parquet spark_catalog.chimcobldhq2ahridtqn081m8cikoi2v9pgmsp314l3morr2c5m46rrie0.customer_master
Output [4]: [CustomerID#756, CustomerName#757, Address#758, Credit_Limit#759]
Batched: true
Location: PreparedDeltaFileIndex [abfss://962bc3af-b163-4e9a-9aa7-6f819f2845b1@onelake.dfs.fabric.microsoft.com/5d568980-3d51-4784-8ad7-aac6f9087f15/Tables/GlobalCorp/customer_master]
PushedFilters: [EqualTo(CustomerID,10001)]
ReadSchema: struct<CustomerID:int,CustomerName:string,Address:string,Credit_Limit:double>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [CustomerID#756, CustomerName#757, Address#758, Credit_Limit#759]

(3) Filter [codegen id : 1]
Input [4]: [CustomerID#756, CustomerName#757, Address#758, Credit_Limit#759]
Condition : (CustomerID#756 = 10001)




In [12]:
txn_df.write.mode("overwrite").format("delta").partitionBy("Transaction_Date").saveAsTable("LH_Nanda.GlobalCorp.Tx_Test")

StatementMeta(, d671fe8f-3537-4656-a708-e2dbe8c8037e, 14, Finished, Available, Finished, False)